human lung cancer

导入相关包

In [29]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [30]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_OmniCell'
save_path = f"/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/ominicell_human_breast_cancer_emb.parquet"


读取嵌入

In [31]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [32]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_OmniCell'

In [33]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    uns: 'neighbors'
    obsm: 'spatial', 'X_OmniCell'
    obsp: 'distances', 'connectivities'

In [34]:
def main(adata,random_seed,res_list,key_added,celltype_col):
    max_value = 0
    metrics = {"method": "OmniCell", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
    save_label_df = None
    for used_res in res_list:
        sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
        true_labels = np.array(adata.obs[celltype_col])
        cluster_labels = np.array(adata.obs[key_added])
        FMI = fowlkes_mallows_score(true_labels, cluster_labels)
        homogeneity = homogeneity_score(true_labels, cluster_labels)
        v_measure = v_measure_score(true_labels, cluster_labels)
        ami = adjusted_mutual_info_score(true_labels, cluster_labels)
        nmi = normalized_mutual_info_score(true_labels, cluster_labels)
        ari = adjusted_rand_score(true_labels, cluster_labels)
        mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

        n_cluster = len(adata.obs[key_added].unique())
        n_celltype = len(adata.obs[celltype_col].unique())
        if mean_value > max_value:
            save_label_df = adata.obs[key_added]
            metrics["ARI"] = ari
            metrics["NMI"] = nmi
            metrics["AMI"] = ami
            metrics["HS"] = homogeneity
            metrics["VM"] = v_measure
            metrics["FMI"] = FMI
            metrics["mean value"] = mean_value
            max_value = mean_value

        print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

    return metrics, save_label_df

In [35]:
metrics, save_label_df = main(adata,random_seed,res_list,key_added,celltype_col)

resolution = 0.1 | n_celltype = 20 | n_cluster = 3 | mean_value = 0.361 | ARI = 0.243 |  NMI = 0.391 | AMI = 0.391 | HS = 0.253 | VM = 0.391 | FMI = 0.499 
resolution = 0.2 | n_celltype = 20 | n_cluster = 6 | mean_value = 0.457 | ARI = 0.344 |  NMI = 0.502 | AMI = 0.502 | HS = 0.4 | VM = 0.502 | FMI = 0.493 
resolution = 0.3 | n_celltype = 20 | n_cluster = 7 | mean_value = 0.527 | ARI = 0.421 |  NMI = 0.575 | AMI = 0.575 | HS = 0.488 | VM = 0.575 | FMI = 0.529 
resolution = 0.4 | n_celltype = 20 | n_cluster = 9 | mean_value = 0.563 | ARI = 0.461 |  NMI = 0.606 | AMI = 0.606 | HS = 0.556 | VM = 0.606 | FMI = 0.541 
resolution = 0.5 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.559 | ARI = 0.445 |  NMI = 0.607 | AMI = 0.607 | HS = 0.562 | VM = 0.607 | FMI = 0.526 
resolution = 0.6 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.566 | ARI = 0.456 |  NMI = 0.611 | AMI = 0.611 | HS = 0.573 | VM = 0.611 | FMI = 0.533 
resolution = 0.7 | n_celltype = 20 | n_cluster = 11 | mean_value

In [36]:
metrics

{'method': 'OmniCell',
 'ARI': 0.4563350029308513,
 'NMI': 0.6109198751552826,
 'AMI': 0.610859500905864,
 'HS': 0.5731619864662989,
 'VM': 0.6109198751552826,
 'FMI': 0.5334140362346576,
 'mean value': 0.5659350461413728}

In [37]:
save_label_df

s1r1_1         3
s1r1_2         3
s1r1_5         1
s1r1_8         3
s1r1_9         3
              ..
s1r2_118748    6
s1r2_118749    6
s1r2_118750    6
s1r2_118751    6
s1r2_118752    6
Name: leiden, Length: 281780, dtype: category
Categories (11, object): ['0', '1', '2', '3', ..., '7', '8', '9', '10']

In [38]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/breast_cancer/ominicell_human_breast_cancer_labels_repeat_{repeat_times}.csv"

In [39]:
save_label_df.to_csv(save_label_df_path)

In [40]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [41]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_{repeat_times}.csv"

In [42]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))